# FULL Combined Brain Tumor MRI Preprocessing Pipeline - Part Wise

**IT2011 - Artificial Intelligence and Machine Learning**  
**Project:** Brain Tumor MRI Classification  
**Dataset:** Brain Tumor Classification MRI  
**Group size:** 6 members

Run the cells from top to bottom in Google Colab.


## Group Members and Preprocessing Responsibilities

| Member | IT Number | Name | Preprocessing Technique |
|---|---|---|---|
| Member 1 | **IT25101776** | **Weerakkody W.W.M.B** | Data Cleaning, Duplicate Removal and Leakage Checking |
| Member 2 | **IT25100233** | **Senanayake S.M.K.D** | Aspect-Ratio Preserving Resizing and Padding |
| Member 3 | **IT25104064** | **Hettiarachchi H.A.A.S** | MRI Image Enhancement using CLAHE |
| Member 4 | **IT25100854** | **Thennakoon T.M.A.N** | Training-Only Data Augmentation |
| Member 5 | **IT25100295** | **Somarathne R.N.N** | Class Imbalance Detection and Training-Only Class Balancing |
| Member 6 | **IT25102346** | **Malintha K.K.I** | Feature Engineering using HOG, GLCM and PCA |


## Project Workflow

1. **Member 1 - IT25101776 - Weerakkody W.W.M.B**  
   Data Cleaning, Duplicate Removal and Leakage Checking.

2. **Common Group Step**  
   Split cleaned Training data into 80% Actual Training and 20% Validation.

3. **Member 2 - IT25100233 - Senanayake S.M.K.D**  
   Aspect-Ratio Preserving Resizing and Padding to 224x224.

4. **Member 3 - IT25104064 - Hettiarachchi H.A.A.S**  
   MRI Image Enhancement using CLAHE.

5. **Common Group Step**  
   Pixel Normalization at model input (`x / 255.0`).

6. **Member 4 - IT25100854 - Thennakoon T.M.A.N**  
   Training-Only Data Augmentation.

7. **Member 5 - IT25100295 - Somarathne R.N.N**  
   Class Imbalance Detection and Training-Only Class Balancing.

8. **Member 6 - IT25102346 - Malintha K.K.I**  
   Feature Engineering using HOG, GLCM and PCA.

9. **Final Group Output**  
   Save processed datasets, feature files, EDA outputs, reports, and downloadable ZIP files.


## Setup - install libraries

In [ ]:
%pip -q install kagglehub opencv-python-headless scikit-image scikit-learn

## Setup - imports and folders

In [ ]:
from pathlib import Path
import os, zipfile, shutil, hashlib, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageEnhance

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE = Path('/content') if Path('/content').exists() else Path.cwd()
WORK = BASE / 'brain_tumor_preprocessing'
DATA_DIR = WORK / 'dataset'
RESULTS = WORK / 'results'
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

## Setup - download the assigned dataset from KaggleHub

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sartajbhuvaji/brain-tumor-classification-mri")

print("Path to dataset files:", path)

DATASET_ROOT = Path(path)
TRAIN_DIR = DATASET_ROOT / "Training"
TEST_DIR = DATASET_ROOT / "Testing"

assert TRAIN_DIR.exists(), f"Training folder not found inside: {DATASET_ROOT}"
assert TEST_DIR.exists(), f"Testing folder not found inside: {DATASET_ROOT}"

print("Training folder:", TRAIN_DIR)
print("Testing folder :", TEST_DIR)


## Setup - create raw manifests

In [ ]:
def make_manifest(root, split_name):
    rows = []
    for class_dir in sorted([p for p in root.iterdir() if p.is_dir()]):
        for p in sorted(class_dir.rglob('*')):
            if p.is_file() and p.suffix.lower() in IMG_EXTS:
                rows.append({
                    'split': split_name,
                    'class_name': class_dir.name,
                    'path': str(p)
                })
    return pd.DataFrame(rows)

train_raw = make_manifest(TRAIN_DIR, 'Training')
test_raw = make_manifest(TEST_DIR, 'Testing')

print('Raw Training images:', len(train_raw))
print('Raw Testing images :', len(test_raw))
display(pd.concat({
    'Training': train_raw['class_name'].value_counts().sort_index(),
    'Testing': test_raw['class_name'].value_counts().sort_index()
}, axis=1).fillna(0).astype(int))

# MEMBER 1 - Data Cleaning, Duplicate Removal and Leakage Checking

**IT Number:** IT25101776  
**Student Name:** Weerakkody W.W.M.B  
**Individual Preprocessing Contribution:** Data Cleaning, Duplicate Removal and Leakage Checking

### Member 1 Identification

**Member:** 1  
**IT Number:** `IT25101776`  
**Name:** **Weerakkody W.W.M.B**  
**Assigned preprocessing technique:** **Data Cleaning, Duplicate Removal and Leakage Checking**

The following cells (`M1 Part ...`) belong to this member's preprocessing contribution.


## M1 Part 1 - EDA before cleaning

In [ ]:
raw_train_counts=train_raw['class_name'].value_counts().sort_index()
display(raw_train_counts.to_frame('images'))
plt.figure(figsize=(8,4)); raw_train_counts.plot(kind='bar')
plt.title('Raw Training Class Distribution'); plt.xlabel('Class'); plt.ylabel('Images'); plt.xticks(rotation=20); plt.tight_layout(); plt.show()

## M1 Part 2 - define audit functions

In [ ]:
def sha256_file(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def image_is_valid(path):
    try:
        with Image.open(path) as im:
            im.verify()
        return True
    except Exception:
        return False

def audit_manifest(df, label):
    out = df.copy()
    out['valid'] = out['path'].map(image_is_valid)
    out['sha256'] = None
    valid_idx = out.index[out['valid']].tolist()
    for n, idx in enumerate(valid_idx, start=1):
        out.at[idx, 'sha256'] = sha256_file(out.at[idx, 'path'])
        if n % 500 == 0:
            print(f'{label}: hashed {n}/{len(valid_idx)}')
    return out

## M1 Part 3 - clean duplicates and exact Train/Test overlap

In [ ]:
train_audit = audit_manifest(train_raw, 'Training')
test_audit = audit_manifest(test_raw, 'Testing')

train_valid = train_audit[train_audit['valid']].copy()
test_valid = test_audit[test_audit['valid']].copy()

# Exact duplicates inside Training: keep one copy.
train_valid['duplicate_in_training'] = train_valid.duplicated('sha256', keep='first')
duplicate_copies = train_valid[train_valid['duplicate_in_training']].copy()
train_unique = train_valid[~train_valid['duplicate_in_training']].copy()

# Exact Training <-> Testing overlap: keep official Test, remove Train copy.
test_hashes = set(test_valid['sha256'].dropna())
train_unique['overlaps_testing'] = train_unique['sha256'].isin(test_hashes)
cross_overlap = train_unique[train_unique['overlaps_testing']].copy()

train_clean = train_unique[~train_unique['overlaps_testing']].copy().reset_index(drop=True)
test_clean = test_valid.copy().reset_index(drop=True)

print('Raw Training                    :', len(train_raw))
print('Exact duplicate copies removed  :', len(duplicate_copies))
print('Unique Training after dedup      :', len(train_unique))
print('Unique Training/Test overlaps    :', len(cross_overlap))
print('Leakage-safe Training            :', len(train_clean))
print('Official Testing kept            :', len(test_clean))

## M1 Part 4 - EDA after cleaning

In [ ]:
clean_counts=train_clean['class_name'].value_counts().sort_index()
clean_compare=pd.concat([raw_train_counts.rename('Before'),clean_counts.rename('After')],axis=1).fillna(0).astype(int)
display(clean_compare)
clean_compare.plot(kind='bar',figsize=(9,4)); plt.title('Before vs After Cleaning'); plt.xlabel('Class'); plt.ylabel('Images'); plt.xticks(rotation=20); plt.tight_layout(); plt.show()

# COMMON STEP - 80% Training / 20% Validation split

## Split Part 1 - create stratified split from cleaned Training only

In [ ]:
from sklearn.model_selection import train_test_split

train_80, validation_20 = train_test_split(
    train_clean,
    test_size=0.20,
    random_state=SEED,
    stratify=train_clean['class_name']
)
train_80 = train_80.reset_index(drop=True)
validation_20 = validation_20.reset_index(drop=True)
official_test = test_clean.reset_index(drop=True)

split_counts = pd.concat({
    'Train_80': train_80['class_name'].value_counts().sort_index(),
    'Validation_20': validation_20['class_name'].value_counts().sort_index(),
    'Official_Test': official_test['class_name'].value_counts().sort_index()
}, axis=1).fillna(0).astype(int)

display(split_counts)
print('Actual Training:', len(train_80))
print('Validation     :', len(validation_20))
print('Official Test  :', len(official_test))

## Split Part 2 - EDA of Train / Validation / Test

In [ ]:
split_counts.plot(kind='bar',figsize=(10,4)); plt.title('Train / Validation / Official Test Distribution'); plt.xlabel('Class'); plt.ylabel('Images'); plt.xticks(rotation=20); plt.tight_layout(); plt.show()

# MEMBER 2 - Aspect-Ratio Preserving Resizing and Padding

**IT Number:** IT25100233  
**Student Name:** Senanayake S.M.K.D  
**Individual Preprocessing Contribution:** Aspect-Ratio Preserving Resizing and Padding

### Member 2 Identification

**Member:** 2  
**IT Number:** `IT25100233`  
**Name:** **Senanayake S.M.K.D**  
**Assigned preprocessing technique:** **Aspect-Ratio Preserving Resizing and Padding**

The following cells (`M2 Part ...`) belong to this member's preprocessing contribution.


## M2 Part 1 - EDA before resizing

In [ ]:
def get_size(path):
    with Image.open(path) as im: return im.size
size_df=train_clean.copy(); size_df['size_tuple']=size_df['path'].map(get_size)
size_df['width']=size_df['size_tuple'].map(lambda x:x[0]); size_df['height']=size_df['size_tuple'].map(lambda x:x[1])
size_df['size']=size_df['width'].astype(str)+' x '+size_df['height'].astype(str)
print('Different original image sizes:',size_df['size'].nunique())
display(size_df['size'].value_counts().head(15).to_frame('count'))
plt.figure(figsize=(10,4)); size_df['size'].value_counts().head(15).plot(kind='bar'); plt.title('Most Common Original MRI Sizes'); plt.xticks(rotation=60,ha='right'); plt.tight_layout(); plt.show()

## M2 Part 2 - define resize and padding

In [ ]:
TARGET_SIZE = (224, 224)
PADDING_VALUE = (0, 0, 0)

def resize_with_padding(image, target_size=TARGET_SIZE, fill=PADDING_VALUE):
    image = image.convert('RGB')
    target_w, target_h = target_size
    original_w, original_h = image.size
    scale = min(target_w / original_w, target_h / original_h)
    new_w = max(1, int(round(original_w * scale)))
    new_h = max(1, int(round(original_h * scale)))
    resized = image.resize((new_w, new_h), Image.Resampling.LANCZOS)

    pad_left = (target_w - new_w) // 2
    pad_top = (target_h - new_h) // 2
    pad_right = target_w - new_w - pad_left
    pad_bottom = target_h - new_h - pad_top

    padded = ImageOps.expand(
        resized,
        border=(pad_left, pad_top, pad_right, pad_bottom),
        fill=fill
    )
    return padded, {
        'original_width': original_w,
        'original_height': original_h,
        'resized_width': new_w,
        'resized_height': new_h,
        'pad_left': pad_left,
        'pad_right': pad_right,
        'pad_top': pad_top,
        'pad_bottom': pad_bottom,
        'final_width': padded.size[0],
        'final_height': padded.size[1],
        'scale_factor': scale,
    }

## M2 Part 3 - visual before/after sample

In [ ]:
sample=train_80.sample(1,random_state=SEED).iloc[0]
with Image.open(sample['path']) as im: original=im.convert('RGB')
stretched=original.resize(TARGET_SIZE,Image.Resampling.LANCZOS); padded,_=resize_with_padding(original)
plt.figure(figsize=(13,4));
plt.subplot(1,3,1); plt.imshow(original); plt.title(f'Original {original.size}'); plt.axis('off')
plt.subplot(1,3,2); plt.imshow(stretched); plt.title('Direct Stretch'); plt.axis('off')
plt.subplot(1,3,3); plt.imshow(padded); plt.title('Resize + Padding'); plt.axis('off')
plt.tight_layout(); plt.show()

## M2 Part 4 - apply resize+padding to all three splits

In [ ]:
GROUP_OUT=RESULTS/'group_pipeline'
MODEL_READY=GROUP_OUT/'model_ready_dataset'
REPORTS=GROUP_OUT/'reports'
EDA_DIR=GROUP_OUT/'eda'
FEATURES=GROUP_OUT/'features'
for p in [MODEL_READY,REPORTS,EDA_DIR,FEATURES]: p.mkdir(parents=True,exist_ok=True)

def resize_save(df,split):
    rows=[]
    for n,row in enumerate(df.itertuples(index=False),start=1):
        src=Path(row.path)
        with Image.open(src) as im: processed,info=resize_with_padding(im)
        dst_dir=MODEL_READY/f'{split}_resized'/row.class_name; dst_dir.mkdir(parents=True,exist_ok=True)
        dst=dst_dir/f'{src.stem}.png'; processed.save(dst)
        rows.append({'class_name':row.class_name,'source_path':str(src),'resized_path':str(dst),**info})
        if n%500==0: print(split,n,'/',len(df))
    return pd.DataFrame(rows)

train_resized=resize_save(train_80,'train'); val_resized=resize_save(validation_20,'validation'); test_resized=resize_save(official_test,'test')
print('All final image sizes:',train_resized[['final_width','final_height']].drop_duplicates().values.tolist())

# MEMBER 3 - MRI Image Enhancement using CLAHE

**IT Number:** IT25104064  
**Student Name:** Hettiarachchi H.A.A.S  
**Individual Preprocessing Contribution:** MRI Image Enhancement using CLAHE

### Member 3 Identification

**Member:** 3  
**IT Number:** `IT25104064`  
**Name:** **Hettiarachchi H.A.A.S**  
**Assigned preprocessing technique:** **MRI Image Enhancement using CLAHE**

The following cells (`M3 Part ...`) belong to this member's preprocessing contribution.


## M3 Part 1 - define CLAHE

In [ ]:
import cv2

CLAHE_CLIP = 2.0
CLAHE_GRID = (8, 8)

def apply_clahe_rgb(pil_image):
    rgb = np.asarray(pil_image.convert('RGB'), dtype=np.uint8)
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_GRID)
    l2 = clahe.apply(l)
    enhanced = cv2.cvtColor(cv2.merge([l2, a, b]), cv2.COLOR_LAB2RGB)
    return Image.fromarray(enhanced)

## M3 Part 2 - EDA before/after CLAHE on one sample

In [ ]:
import cv2
before_img=padded; after_img=apply_clahe_rgb(before_img)
before_gray=cv2.cvtColor(np.asarray(before_img),cv2.COLOR_RGB2GRAY); after_gray=cv2.cvtColor(np.asarray(after_img),cv2.COLOR_RGB2GRAY)
plt.figure(figsize=(12,4));
plt.subplot(1,2,1); plt.imshow(before_img); plt.title('Before CLAHE'); plt.axis('off')
plt.subplot(1,2,2); plt.imshow(after_img); plt.title('After CLAHE'); plt.axis('off'); plt.tight_layout(); plt.show()
plt.figure(figsize=(8,4)); plt.hist(before_gray.ravel(),bins=50,alpha=.55,label='Before'); plt.hist(after_gray.ravel(),bins=50,alpha=.55,label='After'); plt.title('Intensity Histogram - Before vs After CLAHE'); plt.legend(); plt.tight_layout(); plt.show()

## M3 Part 3 - apply CLAHE to resized Train/Validation/Test

In [ ]:
def clahe_from_resized(manifest,split):
    rows=[]
    for n,row in enumerate(manifest.itertuples(index=False),start=1):
        src=Path(row.resized_path)
        with Image.open(src) as im: enhanced=apply_clahe_rgb(im)
        dst_dir=MODEL_READY/split/row.class_name; dst_dir.mkdir(parents=True,exist_ok=True)
        dst=dst_dir/f'{src.stem}.png'; enhanced.save(dst)
        rows.append({'class_name':row.class_name,'source_path':row.source_path,'processed_path':str(dst)})
        if n%500==0: print(split,n,'/',len(manifest))
    return pd.DataFrame(rows)

train_pre=clahe_from_resized(train_resized,'train_unbalanced'); val_pre=clahe_from_resized(val_resized,'validation'); test_pre=clahe_from_resized(test_resized,'test')

# COMMON STEP - Pixel Normalization

## Normalization Part 1 - demonstrate 0-255 to 0-1

In [ ]:
with Image.open(train_pre.iloc[0]['processed_path']) as im: arr=np.asarray(im,dtype=np.uint8)
normalized=arr.astype(np.float32)/255.0
print('Before normalization:',arr.min(),'to',arr.max(),arr.dtype)
print('After normalization :',normalized.min(),'to',normalized.max(),normalized.dtype)

# MEMBER 4 - Training-Only Data Augmentation

**IT Number:** IT25100854  
**Student Name:** Thennakoon T.M.A.N  
**Individual Preprocessing Contribution:** Training-Only Data Augmentation

### Member 4 Identification

**Member:** 4  
**IT Number:** `IT25100854`  
**Name:** **Thennakoon T.M.A.N**  
**Assigned preprocessing technique:** **Training-Only Data Augmentation**

The following cells (`M4 Part ...`) belong to this member's preprocessing contribution.


## M4 Part 1 - EDA before augmentation

In [ ]:
before_aug=train_pre['class_name'].value_counts().sort_index(); display(before_aug.to_frame('images'))
before_aug.plot(kind='bar',figsize=(8,4)); plt.title('Training Counts - Before Augmentation'); plt.xticks(rotation=20); plt.tight_layout(); plt.show()

## M4 Part 2 - define augmentation

In [ ]:
def augment_pil(image, rng):
    image = image.convert('RGB')
    if rng.random() < 0.5:
        image = ImageOps.mirror(image)

    angle = rng.uniform(-12, 12)
    image = image.rotate(
        angle,
        resample=Image.Resampling.BILINEAR,
        fillcolor=(0, 0, 0)
    )

    max_shift = 10
    shift_x = rng.randint(-max_shift, max_shift)
    shift_y = rng.randint(-max_shift, max_shift)
    image = image.transform(
        image.size,
        Image.Transform.AFFINE,
        (1, 0, shift_x, 0, 1, shift_y),
        resample=Image.Resampling.BILINEAR,
        fillcolor=(0, 0, 0)
    )

    image = ImageEnhance.Brightness(image).enhance(rng.uniform(0.90, 1.10))
    image = ImageEnhance.Contrast(image).enhance(rng.uniform(0.90, 1.10))
    return image

## M4 Part 3 - visualize augmented examples

In [ ]:
with Image.open(train_pre.iloc[0]['processed_path']) as im: base_aug=im.convert('RGB')
plt.figure(figsize=(12,8)); plt.subplot(2,3,1); plt.imshow(base_aug); plt.title('Original'); plt.axis('off')
for i in range(5):
    aug=augment_pil(base_aug,random.Random(SEED+i)); plt.subplot(2,3,i+2); plt.imshow(aug); plt.title(f'Augmented {i+1}'); plt.axis('off')
plt.tight_layout(); plt.show()

## M4 Part 4 - EDA after equal augmentation (demonstration)

In [ ]:
after_aug=before_aug*2
pd.concat([before_aug.rename('Before'),after_aug.rename('After equal augmentation')],axis=1).plot(kind='bar',figsize=(9,4)); plt.title('Before vs After Equal Augmentation'); plt.xticks(rotation=20); plt.tight_layout(); plt.show()
print('Equal augmentation increases variety but does not fix the class imbalance.')

# MEMBER 5 - Class Imbalance Detection and Training-Only Class Balancing

**IT Number:** IT25100295  
**Student Name:** Somarathne R.N.N  
**Individual Preprocessing Contribution:** Class Imbalance Detection and Training-Only Class Balancing

### Member 5 Identification

**Member:** 5  
**IT Number:** `IT25100295`  
**Name:** **Somarathne R.N.N**  
**Assigned preprocessing technique:** **Class Imbalance Detection and Training-Only Class Balancing**

The following cells (`M5 Part ...`) belong to this member's preprocessing contribution.


## M5 Part 1 - EDA before balancing

In [ ]:
before_balance=train_pre['class_name'].value_counts().sort_index(); display(before_balance.to_frame('images'))
print('Largest/smallest ratio:',round(float(before_balance.max()/before_balance.min()),3))
before_balance.plot(kind='bar',figsize=(8,4)); plt.title('Actual Training - Before Class Balancing'); plt.xticks(rotation=20); plt.tight_layout(); plt.show()

## M5 Part 2 - calculate target per class

In [ ]:
target_count=int(before_balance.max()); needed=(target_count-before_balance).astype(int)
display(pd.DataFrame({'current':before_balance,'extra_needed':needed})); print('Target per class:',target_count)

## M5 Part 3 - create balanced Training dataset only

In [ ]:
BAL_ROOT=MODEL_READY/'train'
records=[]; classes=sorted(train_pre['class_name'].unique())
for class_idx,cls in enumerate(classes):
    class_df=train_pre[train_pre['class_name']==cls].reset_index(drop=True)
    dst_dir=BAL_ROOT/cls; dst_dir.mkdir(parents=True,exist_ok=True)
    prepared=[]
    for _,row in class_df.iterrows():
        src=Path(row['processed_path']); dst=dst_dir/f'{src.stem}_original.png'; shutil.copy2(src,dst)
        prepared.append(src); records.append({'class_name':cls,'type':'original','output_path':str(dst)})
    extra=target_count-len(prepared)
    for j in range(extra):
        src=prepared[j%len(prepared)]
        with Image.open(src) as im: img=im.convert('RGB')
        aug=augment_pil(img,random.Random(SEED+(class_idx+1)*100000+j))
        dst=dst_dir/f'{src.stem}_balance_aug_{j+1:04d}.png'; aug.save(dst)
        records.append({'class_name':cls,'type':'balance_augmented','output_path':str(dst)})
    print(cls,'added',extra,'final',target_count)
balance_report=pd.DataFrame(records)

## M5 Part 4 - EDA after balancing

In [ ]:
after_balance=balance_report['class_name'].value_counts().sort_index(); compare_bal=pd.concat([before_balance.rename('Before'),after_balance.rename('After')],axis=1); display(compare_bal)
compare_bal.plot(kind='bar',figsize=(9,4)); plt.title('Training - Before vs After Class Balancing'); plt.xticks(rotation=20); plt.tight_layout(); plt.show()
print('Validation unchanged:',len(val_pre),'Official Test unchanged:',len(test_pre))

# MEMBER 6 - Feature Engineering using HOG, GLCM and PCA

**IT Number:** IT25102346  
**Student Name:** Malintha K.K.I  
**Individual Preprocessing Contribution:** Feature Engineering using HOG, GLCM and PCA

### Member 6 Identification

**Member:** 6  
**IT Number:** `IT25102346`  
**Name:** **Malintha K.K.I**  
**Assigned preprocessing technique:** **Feature Engineering using HOG, GLCM and PCA**

The following cells (`M6 Part ...`) belong to this member's preprocessing contribution.


## M6 Part 1 - define HOG + GLCM features

### Member 6 compatibility fix
This version creates a **writable C-contiguous NumPy grayscale array** before calling `graycomatrix()`. This fixes the `ValueError: buffer source array is read-only` error seen with some recent Colab / Python / scikit-image versions.

In [ ]:
from skimage.feature import hog, graycomatrix, graycoprops
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np
from PIL import Image

FEATURE_SIZE = (64, 64)

def feature_vector(path):
    # 1. Open the processed MRI, convert to grayscale, and resize to 64x64.
    with Image.open(path) as im:
        gray_img = im.convert("L").resize(
            FEATURE_SIZE,
            Image.Resampling.LANCZOS
        )

        # IMPORTANT FIX:
        # np.asarray(PIL_image) may return a read-only NumPy array.
        # graycomatrix() needs a writable C-contiguous array in some
        # scikit-image / Python versions.
        gray = np.array(
            gray_img,
            dtype=np.uint8,
            copy=True,
            order="C"
        )

    # Extra safety: guarantee the array is writable and contiguous.
    gray = np.ascontiguousarray(gray).copy()
    gray.setflags(write=True)

    # 2. HOG - edge and shape/orientation features.
    h = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True
    ).astype(np.float32)

    # 3. GLCM - texture relationship matrix.
    g = graycomatrix(
        gray,
        distances=[1],
        angles=[
            0,
            np.pi / 4,
            np.pi / 2,
            3 * np.pi / 4
        ],
        levels=256,
        symmetric=True,
        normed=True
    )

    # 4. GLCM texture properties.
    t = np.array(
        [
            graycoprops(g, "contrast").mean(),
            graycoprops(g, "dissimilarity").mean(),
            graycoprops(g, "homogeneity").mean(),
            graycoprops(g, "energy").mean(),
            graycoprops(g, "correlation").mean()
        ],
        dtype=np.float32
    )

    # 5. Combine HOG + GLCM into one feature vector.
    return np.concatenate([h, t])


# Test the fixed feature extraction before extracting the full matrices.
sample_path = train_pre.iloc[0]["processed_path"]
sample_features = feature_vector(sample_path)

print("Sample image:", sample_path)
print("Raw HOG + GLCM feature count:", len(sample_features))
print("First 10 features:", sample_features[:10])
print("Array fix successful - HOG + GLCM feature extraction works.")


## M6 Part 2 - extract feature matrices

In [ ]:
def matrix(manifest,path_col,label):
    X=[]
    for n,p in enumerate(manifest[path_col],start=1):
        X.append(feature_vector(p))
        if n%500==0: print(label,n,'/',len(manifest))
    return np.vstack(X)

Xtr=matrix(balance_report,'output_path','Balanced Training'); Xva=matrix(val_pre,'processed_path','Validation'); Xte=matrix(test_pre,'processed_path','Testing')
ytr=balance_report['class_name'].to_numpy(); yva=val_pre['class_name'].to_numpy(); yte=test_pre['class_name'].to_numpy()

## M6 Part 3 - fit scaler and PCA on Training only

In [ ]:
scaler=StandardScaler(); Xtr_s=scaler.fit_transform(Xtr); Xva_s=scaler.transform(Xva); Xte_s=scaler.transform(Xte)
pca=PCA(n_components=0.95,svd_solver='full'); Xtr_p=pca.fit_transform(Xtr_s); Xva_p=pca.transform(Xva_s); Xte_p=pca.transform(Xte_s)
print('Before PCA:',Xtr.shape[1]); print('After PCA:',Xtr_p.shape[1]); print('Variance retained:',round(float(pca.explained_variance_ratio_.sum())*100,2),'%')

## M6 Part 4 - EDA after PCA

In [ ]:
cum=np.cumsum(pca.explained_variance_ratio_); plt.figure(figsize=(8,4)); plt.plot(np.arange(1,len(cum)+1),cum); plt.axhline(.95,linestyle='--',label='95%'); plt.title('PCA Cumulative Explained Variance'); plt.xlabel('Components'); plt.ylabel('Cumulative variance'); plt.legend(); plt.tight_layout(); plt.show()
plt.figure(figsize=(8,6));
for cls in sorted(np.unique(ytr)):
    m=ytr==cls; plt.scatter(Xtr_p[m,0],Xtr_p[m,1],s=14,alpha=.55,label=cls)
plt.title('Training Data in PCA Feature Space'); plt.xlabel('PC1'); plt.ylabel('PC2'); plt.legend(); plt.tight_layout(); plt.show()

## M6 Part 5 - save PCA feature files

In [ ]:
def save_pca(X,labels,paths,name):
    df=pd.DataFrame(X,columns=[f'PC{i+1}' for i in range(X.shape[1])]); df.insert(0,'class_name',labels); df.insert(0,'source_path',list(paths)); df.to_csv(FEATURES/name,index=False); return df
save_pca(Xtr_p,ytr,balance_report['output_path'],'train_pca_features.csv'); save_pca(Xva_p,yva,val_pre['processed_path'],'validation_pca_features.csv'); save_pca(Xte_p,yte,test_pre['processed_path'],'test_pca_features.csv')

# FINAL EXPORTS

## Export Part 1 - save manifests and summary

In [ ]:
train_pre.to_csv(REPORTS/'train_unbalanced_processed_manifest.csv',index=False); val_pre.to_csv(REPORTS/'validation_processed_manifest.csv',index=False); test_pre.to_csv(REPORTS/'test_processed_manifest.csv',index=False); balance_report.to_csv(REPORTS/'balanced_training_manifest.csv',index=False)
summary={'raw_training':len(train_raw),'raw_testing':len(test_raw),'duplicate_copies_removed':len(duplicate_copies),'train_test_overlaps_removed_from_training':len(cross_overlap),'clean_training':len(train_clean),'train_80':len(train_80),'validation_20':len(validation_20),'official_test':len(official_test),'balanced_training':len(balance_report),'image_size':[224,224],'normalization':'x/255 at model input'}
with open(GROUP_OUT/'pipeline_summary.json','w') as f: json.dump(summary,f,indent=2)
display(pd.DataFrame(list(summary.items()),columns=['Item','Value']))

## Export Part 2 - create final ZIP files

In [ ]:
def make_zip(folder, zip_path):
    folder = Path(folder)
    zip_path = Path(zip_path)
    if zip_path.exists():
        zip_path.unlink()
    shutil.make_archive(str(zip_path.with_suffix('')), 'zip', root_dir=folder)
    return zip_path

def download_file(path):
    try:
        from google.colab import files
        files.download(str(path))
    except Exception:
        print('Saved at:', path)

In [ ]:
# Remove intermediate resized-only folders from final model-ready package if desired.
for name in ['train_resized','validation_resized','test_resized','train_unbalanced']:
    p=MODEL_READY/name
    if p.exists(): shutil.rmtree(p)

DATASET_ZIP=make_zip(MODEL_READY,WORK/'FINAL_PREPROCESSED_DATASET.zip')
REPORT_ZIP=make_zip(GROUP_OUT,WORK/'ALL_PREPROCESSING_REPORTS_AND_FEATURES.zip')
print(DATASET_ZIP); print(REPORT_ZIP)

## Export Part 3 - download final files

In [ ]:
from google.colab import files
files.download(str(DATASET_ZIP))
files.download(str(REPORT_ZIP))